In [ ]:
import os
# os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "4,5,6,7"

from transformers import AutoModelForCausalLM, AutoTokenizer
import numpy as np
import torch
from tqdm import tqdm

import sys
import string

sys.path.append('../utils/')
import config

sys.path.append('../data/')

sys.path.append('../')
# model = AutoModelForCausalLM.from_pretrained("mistralai/Mistral-7B-v0.1", low_cpu_mem_usage=True, torch_dtype=torch.float16,
#                                                  trust_remote_code=True).cuda()
# tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1")

model = AutoModelForCausalLM.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1", torch_dtype=torch.float16).cuda()
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1", use_fast=False)

In [ ]:
import pandas as pd

df = pd.read_csv('self_generated_data/tqa_mcq.csv')

from datasets import load_dataset

dataset = load_dataset("truthful_qa", "multiple_choice")

In [ ]:
from utils.data_utils import load_truthfulqa_mcq_template
template_path = '../data/truthful-qa' # '../data/hh-rlhf'
template = load_truthfulqa_mcq_template(template_path)
fschat = template['fschat']
print(fschat)

In [ ]:
from utils.inference import vanila_inference, StopOnTokens
from functools import partial

max_new_tokens = 500
inference_fun = partial(vanila_inference, fschat=fschat, max_new_tokens=max_new_tokens)

In [ ]:
def convert_to_quesion_str(q, choices, instruction = None):
    mcq_alphabets = list(string.ascii_lowercase)[:len(choices)]
    mcq_alphabets = [c.upper() for c in mcq_alphabets]
    choice_str = " ".join(["("+ mcq_alphabets[i] + ")" + " " + c for i, c in enumerate(choices)])
    if not instruction:
        question_str = f"Question: {q}\nOptions: {choice_str}\nChoose the correct option. The answer is  "
    else:
        question_str = f"Human: Choose the correct option to answer the question. Question: {q}\nOptions: {choice_str} {instruction}\nAssistant: "
    return question_str, choice_str

In [ ]:
import json
outdir = 'results/instruct_model'

pred_obj_all = []

for row_id, row in tqdm(df.iterrows()):
    print(f'########## {row_id} ##########')
    q = row['question']
    labels = row['new_labels'].split(',')
    labels = [l.strip().rstrip() for l in labels]
    labels[0] = labels[0][-1]
    labels[-1] = labels[-1][0]

    print(row['shuffled_options_idx'])
    shuffled_options_idx = row['shuffled_options_idx'].split(',')
    shuffled_options_idx = [l.strip().rstrip() for l in shuffled_options_idx]
    shuffled_options_idx[0] = shuffled_options_idx[0][-1]
    shuffled_options_idx[-1] = shuffled_options_idx[-1][0]
    shuffled_options_idx = np.array([int(l) for l in shuffled_options_idx])

    choices_orig = dataset['validation'][row_id]['mc1_targets']['choices']
    choices_shuffled = np.array(choices_orig)[shuffled_options_idx].tolist()
    
    labels = np.array([int(l) for l in labels])
    gt = np.argwhere(labels == 1).flatten()[0]
    question_str, choices_str = convert_to_quesion_str(q, choices_shuffled)
    print(question_str)
    ans = inference_fun(raw_query=question_str, model=model, tokenizer=tokenizer).split(question_str)[-1].strip().rstrip()
 
    print(ans)

    tmp = {'question': q,
           'answer': ans,
           'label': str(gt),
           'choices': choices_str,
           'n_ans': len(labels)
          }
    
    pred_obj_all.append(tmp)

    if not os.path.exists(os.path.join(outdir)):
        os.makedirs(os.path.join(outdir))
    with open('{}/tqa_mcq_{}_res_{}.json'.format(outdir, 'mistral@7b', row_id), 'w') as f:
        f.write(json.dumps(tmp))
    

In [ ]:
import re

def parse_answer(pred_text):
    pattern = re.compile(r'\([A-Z]\)')
    res = pattern.findall(pred_text)
    if len(res) == 1:
        answer = res[0][1]  # 'A', 'B', ...
    else:
        answer = "FAILED"
    return answer

In [ ]:
f.close()

In [ ]:
pred_obj_all[0]

In [ ]:
def get_pred(pred_obj):
    ans = pred_obj['answer']
    options = list(string.ascii_lowercase)[:pred_obj['n_ans']]
    options = [o.upper() for o in options]
    # print(options)
    pred = parse_answer(pred_obj['answer'])
    # print(pred)
    if pred != "FAILED":
        pred_idx = np.argwhere(np.array(options)==pred).flatten()[0]
    else:
        arr_wrong_choice=np.arange(pred_obj['n_ans']).tolist()
        arr_wrong_choice.pop(int(pred_obj['label']))
        pred_idx = np.random.choice(arr_wrong_choice)
    return pred_idx

In [ ]:
preds_all = []
labels_all = []
for obj_ in pred_obj_all:
    pred_idx = get_pred(obj_)
    preds_all.append(pred_idx)
    labels_all.append(int(obj_['label']))

In [ ]:
(np.array(preds_all)==np.array(labels_all)).mean()